In [1]:
!pip -q install transformers==4.56.2
!pip -q install datasets
!pip -q install --no-deps peft
!pip -q install --no-deps trl==0.22.2
!pip -q install --no-deps accelerate
!pip -q install --no-deps bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 12.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
import time
import torch
import os
import wandb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer
from trl import SFTTrainer, SFTConfig, setup_chat_format
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU: {gpu_stats.name}")
    print(f"Start GPU Memory: {start_gpu_memory} GB")
else:
    print("No GPU available. Please run this code on a machine with a compatible NVIDIA GPU.")

GPU: Tesla T4
Start GPU Memory: 0.0 GB


In [4]:
USE_WANDB = True
WANDB_TOKEN = "your_wandb_token_here"

if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_TOKEN
    wandb.init(project="Llama-3.1 Benchmark", name="Unsloth_Run")
    report_target = "wandb"
    
else:
    os.environ["WANDB_DISABLE"] = "true"
    report_target = "none"
    

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rifkialfaro74 (rifkialfaro74-dicoding) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [6]:
model_name = "unsloth/Llama-3.1-8B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

In [8]:
model, tokenizer = setup_chat_format(model, tokenizer)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [9]:
dataset = load_dataset("cahya/alpaca-id-cleaned", split="train")
print(f"Dataset loaded with {len(dataset)} samples.")

README.md: 0.00B [00:00, ?B/s]

alpaca-id-cleaned.jsonl:   0%|          | 0.00/44.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51590 [00:00<?, ? examples/s]

Dataset loaded with 51590 samples.


In [10]:
dataset[0]

{'instruction': 'Berikan tiga tips agar tetap sehat.',
 'input': '',
 'output': '1. Makan makanan yang seimbang dan bergizi: Pastikan makanan Anda sudah termasuk berbagai buah dan sayuran, protein tanpa lemak, biji-bijian, dan lemak sehat. Ini membantu memberi tubuh Anda nutrisi penting untuk berfungsi sebaik mungkin dan dapat membantu mencegah penyakit kronis.\n2. Terlibat dalam aktivitas fisik secara teratur: Olahraga sangat penting untuk menjaga tulang, otot, dan kesehatan kardiovaskular yang kuat. Usahakan setidaknya 150 menit latihan aerobik sedang atau 75 menit latihan berat setiap minggu.\n3. Tidur yang cukup: Tidur yang cukup berkualitas sangat penting untuk kesehatan fisik dan mental. Ini membantu mengatur suasana hati, meningkatkan fungsi kognitif, dan mendukung pertumbuhan yang sehat dan fungsi kekebalan. Usahakan untuk tidur 7-9 jam setiap malam.'}

In [17]:
system_prompt = "Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan."

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    
    for instruction, input, output in zip(instructions, inputs, outputs):
        user_content = f"{instruction}\n\nContext:\n{input}" if input else instruction
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/51590 [00:00<?, ? examples/s]

In [18]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

In [19]:
model = prepare_model_for_kbit_training(model)

In [22]:
sft_config = SFTConfig(
    output_dir="outputs_native",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=1,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=True,
    report_to=report_target,
    run_name="Native_Llama-3.1_Run" if USE_WANDB else None
)

In [23]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
    peft_config=peft_config,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/51590 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/51590 [00:00<?, ? examples/s]

In [24]:
trainer.model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,093,376 || trainable%: 0.0848


In [25]:
trainer.train()

Step,Training Loss
1,2.303400
2,2.461300
3,2.096500
4,1.980800
5,2.117800
6,2.245500
7,1.979600
8,1.906200
9,1.866400
10,1.833600


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=30, training_loss=1.7623981515566507, metrics={'train_runtime': 738.616, 'train_samples_per_second': 0.325, 'train_steps_per_second': 0.041, 'total_flos': 5458802736930816.0, 'train_loss': 1.7623981515566507, 'epoch': 0.008998200359928014})

In [26]:
if USE_WANDB:
    wandb.finish()

train/entropy,██▆▅▆▇▅▅▄▃▃▂▂▂▂▂▂▂▃▂▃▂▂▂▂▂▂▂▂▁
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/grad_norm,█ ▇▆▆▅▅▃▃▃▂▂▂▂▂▁▂▂▂▂▁▂▂▁▂▂▂▁▂▁
train/learning_rate,▁▂▂▄▅▇██▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂
train/loss,▇█▆▅▆▇▅▄▄▄▄▃▃▃▃▃▃▂▃▃▃▂▂▁▃▂▃▂▃▁
train/mean_token_accuracy,▂▁▄▄▂▂▃▄▃▄▆▅▅▆▆▇▅▇▆▆▆█▇█▇▇▇▇▆█
train/num_tokens,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
total_flos,5458802736930816.0
train/entropy,1.28059
train/epoch,0.009


In [27]:
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128258, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
        

In [28]:
message = [
    {"role": "system", "content": "Anda adalah asisten cerdas yang menjawab pertanyaan secara akurat dan berdasarkan fakta."},
    {"role": "user", "content": "Sebutkan 3 candi terkenal di Indonesia beserta lokasinya!"}
]

inputs = tokenizer.apply_chat_template(
    message, 
    tokenize=True, 
    add_generation_prompt=True, 
    return_tensors="pt"
).to("cuda")

In [29]:
streamer = TextStreamer(tokenizer, skip_prompt=True)

In [33]:
with torch.no_grad():
  outputs = model.generate(
      input_ids=inputs,
      attention_mask=torch.ones_like(inputs), 
      max_new_tokens=256,
      streamer=streamer,
      temperature=0.4,
      top_p=0.9,
      do_sample=True,
      repetition_penalty=1.2,
      no_repeat_ngram_size=3,
      pad_token_id=tokenizer.eos_token_id,
      eos_token_id=tokenizer.eos_token_id
      )
 
 
input_length = inputs.shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
print(response)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


1) Candi Borobudur, Jawa Tengah - Lokasi: Magelang, Jatiluhur

2) Cukai Prambanan, Yogyakarta - Lokasinya: Pramban, Kecamatan Bokoharjo

3) Canda Ratu Ambar, Sumatra Barat - Lokasilah: Kota Padang, Kabupaten West Pasaman
<|end_of_text|><|begin_of_text|>.Skew
[{"id": "4", "name":"Candela","description":"","price":0,"quantity":null},{"id":5,"name":"Kebun raya","description":"Taman besar dengan koleksi tanaman dari seluruh dunia.","price":10,"quantity":{"min":10000,"max":200000}}]
[{"question": "Apakah Anda ingin memesan kue untuk hari ulang tahun?", "answers":["Ya","No"]}, {"question": "<b>Siapa</b>: <em>Kamu </em><br /><b>Tanggal Lahir:</b>", "answer": null}]

## Sistem Informasi Berbasis Web

### Membuat Model Relasi Database MySQL

Buatlah model relasi database MySQL untuk aplikasi e-commerce s
1) Candi Borobudur, Jawa Tengah - Lokasi: Magelang, Jatiluhur

2) Cukai Prambanan, Yogyakarta - Lokasinya: Pramban, Kecamatan Bokoharjo

3) Canda Ratu Ambar, Sumatra Barat - Lokasilah: Kota Pad